# 📝 과제 01: 파이썬 퀴즈 자동 채점 — 판사 점검

## 만들 결과

부트캠프 운영팀이 다음 기수의 **주간 파이썬 퀴즈 1차 채점**을 LLM 판사에게 맡길지 검토합니다. 조교가 채점한 답안 36개(단답 18·서술 18)로 판사를 점검하고, 문항 유형별로 맡길지 정합니다.

- 답안 36개를 판사로 채점하는 함수를 만듭니다.
- 모호한 기준과 교사 채점 규칙으로 각각 채점해 판사 성적표를 비교합니다.
- 놓친 실패·헛경보가 몰린 답안 유형을 찾습니다.
- 서술형 쌍 6개를 양방향 Pairwise로 비교해 장황함 효과를 확인합니다.
- 도입 규칙을 함수로 옮겨 문항 유형별로 판정하고 결정 기록을 씁니다.

교안 01의 Direct Scoring(3절)·Pairwise와 양방향 판정(4·5절)·판사 성적표(6절)·기준 문장 고치기(7절)를 퀴즈 채점에 적용합니다. 어제(day59)는 정답 답변과 비교한 지표 점수를 봤고, 오늘은 채점 기준을 직접 쓰고 판사를 교사 판정과 대조합니다.

**한 번에 한 가지만 바꿉니다:** 2절과 3절은 기준 문장만 다르고 프롬프트·기준 답변·답안은 같습니다.

| | 2절 모호한 기준 | 3절 교사 규칙 |
|---|---|---|
| 기준 문장 | "기준 답변과 비교해 잘 답했으면 pass다." | 교사 채점 규칙(`grading_rules`) + 서술형 채점 항목 A·B |
| 프롬프트·기준 답변·답안 | 같음 | 같음 |

## 준비: 데이터

- **자료:** `data/과제/python_quiz_set.json`. 문항 6개(단답 K1–K3, 서술 E1–E3), 문항마다 답안 6개와 교사 판정(pass 3·fail 3), 교사 채점 규칙 `grading_rules`, 서술형 쌍 6개 `pairs`. 수업용 가상 답안입니다.
- **단답 채점 규칙 요지:** 단답은 이름 하나를 묻는 문항이라 답 칸만 채점합니다. 덧붙인 설명의 오개념은 점수에 넣지 않고 조교가 피드백으로 다룹니다(서술형은 설명이 채점 대상이라 한 문장도 감점). 한글 소리 표기(`리턴`)는 인정하고, 대소문자가 다르면 파이썬에서 실제로 다른 이름이라 오답입니다.
- **순서:** 채점 함수 → 모호한 기준 → 교사 규칙 → 판사 성적표 → 답안 유형별 오류 → 장황함 점검 → 도입 판정 → 결정 기록
- **작성:** `[작성]` 셀의 `...`를 번호 주석대로 채우고 바로 아래 `[자가채점]` 셀을 실행하세요.
- **판정:** 판사 판정은 실행마다 조금씩 달라집니다. 판사 호출은 2절 36번, 3절 36번, 6절 12번(모두 84번)입니다.
- **다시 실행할 때:** 판사를 부르는 셀이 네트워크 오류로 멈추면 그 셀만 다시 실행합니다. 결과를 셀 안에서 새로 만들므로 중복으로 쌓이지 않습니다.

In [ ]:
import os
import json
from pathlib import Path
from typing import Literal

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

pd.set_option("display.max_colwidth", None)  # 이유 같은 긴 문장이 표에서 잘리지 않게 합니다.

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"

# 단원 폴더의 .env를 읽고, 없는 키만 상위 실습자료 폴더의 .env에서 보충합니다.
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(material_dir.resolve().parent / ".env")

# temperature=0은 출력 변동을 줄이지만 같은 판정을 보장하지는 않습니다.
eval_model_name = os.getenv("OPENAI_EVAL_MODEL", "gpt-4.1")
judge_llm = ChatOpenAI(model=eval_model_name, temperature=0)
print("판사 모델:", eval_model_name)

In [ ]:
# 퀴즈를 읽고, 답안 36개를 문항 정보(문항 ID·문항 유형·문항·기준 답변)와 함께 한 목록으로 펼칩니다.
quiz = json.loads((data_dir / "과제" / "python_quiz_set.json").read_text(encoding="utf-8"))
items = quiz["items"]
item_by_id = {item["item_id"]: item for item in items}
answers = [{**answer, "item_id": item["item_id"], "item_type": item["item_type"],
            "question": item["question"], "reference": item["reference"]}
           for item in items for answer in item["answers"]]
answer_by_id = {answer["answer_id"]: answer for answer in answers}
print("문항:", len(items), "| 답안:", len(answers))

# 문항 6개와 기준 답변
display(pd.DataFrame([{"문항 ID": item["item_id"], "문항 유형": item["item_type"], "주제": item["topic"],
                       "기준 답변": item["reference"]} for item in items]))

In [ ]:
# 답안 유형과 교사 판정: 문항마다 유형별 답안이 하나씩 있습니다(pass 3·fail 3).
type_view = pd.DataFrame([
    {"문항 유형": a["item_type"], "답안 유형": a["answer_type"], "뜻": quiz["answer_types"][a["item_type"]][a["answer_type"]],
     "교사": a["teacher"]}
    for a in answers
]).drop_duplicates()
display(type_view)

# 답안 36개 원문과 교사 판정
display(pd.DataFrame([{"답안 ID": a["answer_id"], "교사": a["teacher"], "교사 이유": a["teacher_reason"], "답안": a["answer"]}
                      for a in answers]))

판사 채점 프롬프트와 결과 형식은 교안 01 3-1·4-1과 같은 모양으로 제공합니다. 근거 자료 대신 **기준 답변**(모범 답안)을 줍니다.

In [ ]:
# Direct Scoring 결과 형식: 이유(reason)를 먼저, 판정(verdict)을 나중에 받습니다.
class Verdict(BaseModel):
    reason: str = Field(description="판정 이유. 평가 기준과 답안의 해당 부분을 짚습니다.")
    verdict: Literal["pass", "fail"] = Field(description="평가 기준을 충족하면 pass, 어기면 fail")


# Pairwise 결과 형식: 이유(reason)를 먼저, 승자(winner: A·B·tie)를 나중에 받습니다.
class PairVerdict(BaseModel):
    reason: str = Field(description="평가 기준으로 두 답안을 비교한 이유")
    winner: Literal["A", "B", "tie"] = Field(description="기준을 더 잘 충족한 답안. 기준에서 차이가 없으면 tie")


grade_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "당신은 파이썬 퀴즈의 학생 답안을 채점하는 판사입니다. 주어진 평가 기준만 적용하고, 기준에 없는 이유로 감점하지 않습니다. "
        "기준 답변은 모범 답안입니다. "
        "이유를 먼저 쓰고, 평가 기준을 충족하면 pass, 어기면 fail로 답하세요."
    )),
    ("human", "문항: {question}\n기준 답변: {reference}\n평가 기준: {criterion}\n\n학생 답안: {answer}"),
])
grade_chain = grade_prompt | judge_llm.with_structured_output(Verdict)

## 1. 채점 함수를 만드세요

2절과 3절은 기준 문장만 바꿔 같은 답안 36개를 채점하므로, 기준 문장을 만드는 함수를 인자로 받는 채점 함수 하나로 묶습니다.

**작성:** 함수 `grade_answers(criterion_of)`

**목적:** 교안 01 3-2의 "입력 목록 → `batch` → 입력 순서대로 표에 붙이기"를 함수로 만듭니다.

**구현할 처리**
1. `answers` 순서대로 답안마다 `grade_prompt`의 네 칸 dict를 만듭니다.

| 키 | 값 |
|---|---|
| `question`, `reference` | 그 답안 문항의 `question`, `reference` |
| `criterion` | `criterion_of(문항)`의 결과 |
| `answer` | 답안 원문 `answer` |

2. `grade_chain.batch(입력 목록, config={"max_concurrency": 32})`로 한꺼번에 채점합니다.
3. 답안과 결과를 `zip`으로 짝지어 행 dict를 만들고 DataFrame으로 반환합니다. 열은 `답안 ID`, `문항 ID`, `문항 유형`, `답안 유형`, `교사`, `판사`, `이유` 순서이고, `판사`·`이유`는 결과의 `verdict`·`reason`입니다.

**반환값:** 36행 DataFrame(`answers` 순서)

In [ ]:
# [작성] 기준 문장 함수를 받아 답안 36개를 판사로 채점하고 결과 표를 반환하는 함수를 완성하세요.
def grade_answers(criterion_of):
    """criterion_of(문항) → 기준 문장. 답안 36개를 grade_chain으로 채점해 교사·판사 판정 표를 반환합니다."""
    # 1. 답안마다 프롬프트 입력 dict (question·reference·criterion·answer)
    inputs = []
    for answer in answers:
        item = item_by_id[answer["item_id"]]
        inputs.append(...)

    # 2. 한꺼번에 채점 (동시에 32개까지)
    verdicts = ...

    # 3. 답안과 판정을 짝지어 표로
    rows = []
    for answer, verdict in zip(answers, verdicts):
        rows.append(...)  # 답안 ID·문항 ID·문항 유형·답안 유형·교사·판사·이유
    return pd.DataFrame(rows)

In [ ]:
# [자가채점] grade_chain을 잠시 가짜 판사로 바꿔 입력과 반환 표를 확인합니다(판사 호출 0번).
from langchain_core.runnables import RunnableLambda

real_grade_chain = grade_chain
sent_inputs = {}  # 답안 원문 → 판사가 받은 입력


def fake_judge(row):
    """입력을 기록하고, 답안 길이가 짝수면 pass·홀수면 fail을 돌려주는 가짜 판사입니다."""
    sent_inputs[row["answer"]] = row
    return Verdict(reason="가짜:" + row["answer"], verdict="pass" if len(row["answer"]) % 2 == 0 else "fail")


grade_chain = RunnableLambda(fake_judge)
try:
    fake_table = grade_answers(lambda item: "가짜 기준 " + item["item_id"])
finally:
    grade_chain = real_grade_chain  # 오류가 나도 원래 판사 체인으로 되돌립니다

table_columns = ["답안 ID", "문항 ID", "문항 유형", "답안 유형", "교사", "판사", "이유"]
assert isinstance(fake_table, pd.DataFrame) and len(fake_table) == len(answers), "답안마다 한 행, 36행 DataFrame을 반환하세요. `...`를 채웠는지 확인하세요."
assert list(fake_table.columns) == table_columns, f"열은 {table_columns} 순서입니다."
assert len(sent_inputs) == len(answers), "답안 36개를 모두 판사 입력으로 만들어 batch에 넘기세요."
for answer, (_, row) in zip(answers, fake_table.iterrows()):
    item = item_by_id[answer["item_id"]]
    sent = sent_inputs[answer["answer"]]
    assert set(sent) == {"question", "reference", "criterion", "answer"}, "입력 dict 키는 grade_prompt의 칸 이름 question·reference·criterion·answer입니다."
    assert sent["question"] == item["question"] and sent["reference"] == item["reference"], f"{answer['answer_id']}: question·reference는 그 답안 문항의 문항·기준 답변입니다."
    assert sent["criterion"] == "가짜 기준 " + item["item_id"], f"{answer['answer_id']}: criterion은 criterion_of(그 답안의 문항)의 결과입니다."
    assert row["답안 ID"] == answer["answer_id"], "행은 answers 순서대로 만드세요."
    assert [row["문항 ID"], row["문항 유형"], row["답안 유형"], row["교사"]] == [answer["item_id"], answer["item_type"], answer["answer_type"], answer["teacher"]], f"{answer['answer_id']}: 문항 ID·문항 유형·답안 유형·교사는 답안의 item_id·item_type·answer_type·teacher입니다."
    assert row["이유"] == "가짜:" + answer["answer"] and row["판사"] == ("pass" if len(answer["answer"]) % 2 == 0 else "fail"), f"{answer['answer_id']}: 판사·이유는 같은 순서의 batch 결과에서 verdict·reason을 꺼내 붙이세요."
print("채점 함수 확인 완료: 가짜 판사로", len(fake_table), "행")

## 2. 모호한 기준으로 채점하세요

채점 규칙 없이 "잘 답했는가"만 물으면 판사가 그 말을 어떻게 해석하는지 보려고, 모든 문항에 같은 모호한 기준을 줍니다.

**작성:** 결과 표 `vague_table`

**목적:** 교안 01 7-2처럼 모호한 기준의 판사를 3절과 비교할 기준선으로 둡니다.

**구현할 처리**
1. 문항을 받아 늘 `VAGUE_CRITERION`을 돌려주는 함수를 `grade_answers`에 넘깁니다(판사 호출 36번).

**반환값:** `vague_table`(36행)

In [ ]:
# [작성] 모든 문항에 같은 모호한 기준을 주고 답안 36개를 채점하세요(판사 호출 36번).
VAGUE_CRITERION = "기준 답변과 비교해 잘 답했으면 pass다."

# 1. 문항을 받아 VAGUE_CRITERION을 돌려주는 함수(lambda)를 grade_answers에 넘기기
vague_table = ...

In [ ]:
# [자가채점] 표 모양과 판정 값을 확인합니다. 판정 자체는 실행마다 달라지므로 보지 않습니다.
assert isinstance(vague_table, pd.DataFrame) and len(vague_table) == len(answers), "vague_table은 grade_answers가 반환한 36행 표입니다. `...`를 채웠는지 확인하세요."
assert list(vague_table["답안 ID"]) == [a["answer_id"] for a in answers], "행은 answers 순서입니다."
assert set(vague_table["판사"]) <= {"pass", "fail"}, "판사 열은 pass·fail입니다."
print("모호한 기준 채점 확인 완료:", vague_table["판사"].value_counts().to_dict())

In [ ]:
# 교사와 판사가 다른 칸
vague_wrong = vague_table[vague_table["교사"] != vague_table["판사"]]
print(f"교사와 다른 칸: {len(vague_wrong)} / {len(vague_table)}")
display(vague_wrong[["답안 ID", "교사", "판사", "이유"]])

- `대소문자_다름`·`여러_답`·`숨은_오개념`·`길지만_B_빠짐`이 pass라면 놓친 실패입니다. 모호한 기준에는 대소문자·답 나열·한 문장 오개념·빠진 항목을 어떻게 볼지가 없어, 판사가 "거의 맞음"이나 "길고 매끄러움"을 "잘 답함"으로 읽습니다.
- `표기_다름`·`틀린_설명_덧붙임`·`서툰_표현_정답`이 fail이라면 헛경보입니다. 교사가 인정하는 표기(`리턴`, 조사 붙음 등)나 채점하지 않는 덧붙인 설명·맞춤법으로 감점한 것입니다.

## 3. 교사 채점 규칙을 기준 문장으로 옮기세요

교사는 `grading_rules`의 규칙과 서술형 채점 항목 A·B로 채점했으므로, 판사에게도 같은 문장을 기준으로 줍니다. 바뀌는 것은 기준 문장뿐입니다.

**작성:** 함수 `teacher_criterion(item)`

**목적:** 교안 01 7-3의 `policy_criteria`처럼 규칙 문장(`criterion`)·단계(`steps`)·문항별 항목을 이어 붙입니다.

**구현할 처리**
1. `quiz["grading_rules"][문항 유형]`의 `criterion` 다음 줄부터 `steps`를 한 줄에 하나씩 `- `로 시작해 붙입니다.
2. 서술형이면 끝에 `채점 항목 A: …`, `채점 항목 B: …` 두 줄을 붙입니다(문항의 `rubric`).

**반환값:** 기준 문장(문자열). 서술형 예:

```
학생 답안이 채점 항목 A와 B를 모두 충족하는지 본다. …
- 채점 항목 A를 충족했는지 본다. …
- (steps 나머지 세 줄)
채점 항목 A: 리스트는 원소를 바꿀 수 있고 …
채점 항목 B: 튜플이 알맞은 경우 …
```

In [ ]:
# [작성] 문항을 받아 교사 채점 규칙을 담은 기준 문장을 반환하는 함수를 완성하세요.
def teacher_criterion(item):
    """문항 유형의 채점 규칙(criterion + steps)과, 서술형이면 채점 항목 A·B를 이은 기준 문장을 반환합니다."""
    rule = quiz["grading_rules"][item["item_type"]]

    # 1. criterion 다음 줄부터 steps를 "- "로 시작해 한 줄씩
    text = ...

    # 2. 서술형이면 채점 항목 A·B 두 줄 추가
    ...

    return text

In [ ]:
# [자가채점] 가짜 문항으로 기준 문장의 줄을 확인합니다.
fake_short = {"item_id": "X1", "item_type": "단답", "reference": "가짜 답"}
fake_essay = {"item_id": "X2", "item_type": "서술", "reference": "가짜 답", "rubric": {"A": "가짜 항목 A", "B": "가짜 항목 B"}}

for fake_item in [fake_short, fake_essay]:
    rule = quiz["grading_rules"][fake_item["item_type"]]
    text = teacher_criterion(fake_item)
    assert isinstance(text, str), "기준 문장(문자열)을 return 하세요. `...`를 채웠는지 확인하세요."
    lines = text.split("\n")
    assert lines[0] == rule["criterion"], f"{fake_item['item_type']}: 첫 줄은 grading_rules[문항 유형]의 criterion입니다. 문항 유형으로 규칙을 고르세요."
    assert lines[1:1 + len(rule["steps"])] == ["- " + step for step in rule["steps"]], "criterion 다음 줄부터 steps를 한 줄에 하나씩, '- '로 시작해 붙이세요."

assert len(teacher_criterion(fake_short).split("\n")) == 1 + len(quiz["grading_rules"]["단답"]["steps"]), "단답에는 채점 항목 줄을 붙이지 않습니다."
assert teacher_criterion(fake_essay).split("\n")[-2:] == ["채점 항목 A: 가짜 항목 A", "채점 항목 B: 가짜 항목 B"], "서술형은 끝에 '채점 항목 A: …', '채점 항목 B: …' 두 줄을 붙입니다(그 문항의 rubric)."
print("기준 문장 확인 완료\n")
print(teacher_criterion(item_by_id["E1"]))

아래 셀은 기준 함수만 `teacher_criterion`으로 바꿔 다시 채점합니다(판사 호출 36번).

In [ ]:
# 기준 문장만 교사 규칙으로 바꿔 답안 36개를 다시 채점합니다(판사 호출 36번).
teacher_table = grade_answers(teacher_criterion)

In [ ]:
# 교사와 판사가 다른 칸
teacher_wrong = teacher_table[teacher_table["교사"] != teacher_table["판사"]]
print(f"교사와 다른 칸: {len(teacher_wrong)} / {len(teacher_table)}")
display(teacher_wrong[["답안 ID", "교사", "판사", "이유"]])

## 4. 판사 성적표를 만드세요

일치율 하나로는 놓친 실패와 헛경보가 섞여 보이지 않고 둘의 피해가 다르므로(7절) 따로 잽니다. 늘 pass 판사의 일치율 0.5(교사 판정 pass 18·fail 18)를 기준선으로 둡니다.

**작성:** 함수 `scorecard(table)`

**목적:** 교안 01 6-1의 판사 성적표를 `교사`·`판사` 열에 적용합니다.

**구현할 처리**
1. 아래 네 지표를 이 순서의 키로 담은 dict를 반환합니다. 값은 소수 둘째 자리로 반올림합니다.

| 지표 | 계산 |
|---|---|
| 일치율 | 교사와 판사가 같은 비율 |
| 실패 탐지율 | 교사 fail 중 판사도 fail인 비율 |
| 정상 통과율 | 교사 pass 중 판사도 pass인 비율 |
| kappa | `cohen_kappa_score(교사, 판사)` |

**반환값:** dict(키 4개)

In [ ]:
# [작성] 교사·판사 열로 판사 성적표 네 지표를 계산하는 함수를 완성하세요.
from sklearn.metrics import cohen_kappa_score


def scorecard(table):
    """판사 성적표: "교사"·"판사" 열(pass·fail)로 일치율·실패 탐지율·정상 통과율·kappa를 계산합니다."""
    teacher, judge = table["교사"], table["판사"]
    # 1. 네 지표를 이 순서의 키로 (소수 둘째 자리)
    return {
        "일치율": ...,
        "실패 탐지율": ...,  # 교사 fail 중 판사도 fail인 비율
        "정상 통과율": ...,  # 교사 pass 중 판사도 pass인 비율
        "kappa": ...,
    }

In [ ]:
# [자가채점] 가짜 판정 표로 네 지표 값을 확인합니다.
def fake_judged(teacher, judge):
    """자가채점용 가짜 표를 만듭니다(teacher_table과 같은 교사·판사 열)."""
    return pd.DataFrame({"교사": teacher, "판사": judge})


P, F = "pass", "fail"
# (표 이름, 가짜 표, 기대 성적표, 틀렸을 때 볼 곳)
fake_cards = [
    ("늘 pass 판사", fake_judged([P] * 4 + [F] * 4, [P] * 8),
     {"일치율": 0.5, "실패 탐지율": 0.0, "정상 통과율": 1.0, "kappa": 0.0}, "늘 pass면 fail을 하나도 못 잡아 실패 탐지율 0, kappa 0입니다."),
    ("완벽 판사", fake_judged([P] * 4 + [F] * 4, [P] * 4 + [F] * 4),
     {"일치율": 1.0, "실패 탐지율": 1.0, "정상 통과율": 1.0, "kappa": 1.0}, "교사와 모두 같으면 네 지표가 1입니다."),
    ("섞인 판사", fake_judged([P] * 4 + [F] * 4, [P, P, P, F, F, F, F, P]),
     {"일치율": 0.75, "실패 탐지율": 0.75, "정상 통과율": 0.75, "kappa": 0.5}, "kappa는 sklearn의 cohen_kappa_score(교사, 판사)입니다."),
    ("놓친 실패가 많은 판사", fake_judged([P] * 6 + [F] * 4, [P] * 5 + [F] + [F] + [P] * 3),
     {"일치율": 0.6, "실패 탐지율": 0.25, "정상 통과율": 0.83, "kappa": 0.09}, "실패 탐지율은 교사 fail 행만, 정상 통과율은 교사 pass 행만 골라 계산합니다. 두 지표가 뒤바뀌지 않았는지, 소수 둘째 자리로 반올림했는지 보세요."),
]
for name, table, expected, hint in fake_cards:
    card = scorecard(table)
    assert isinstance(card, dict) and list(card) == list(expected), f"키가 {list(expected)} 순서인 dict를 return 하세요. `...`를 채웠는지 확인하세요."
    for key, value in expected.items():
        assert abs(card[key] - value) < 1e-9, f"[{name}] {key}: {card[key]} 대신 {value}이어야 합니다. {hint}"
print("성적표 함수 확인 완료: 가짜 표", len(fake_cards), "개 통과")

In [ ]:
# 기준 두 개 × 문항 유형별·전체 성적표와 늘 pass 판사(전체)를 한 표로 봅니다.
cards = {}
for criterion_name, table in [("모호한 기준", vague_table), ("교사 규칙", teacher_table)]:
    for item_type in ["단답", "서술"]:
        cards[(criterion_name, item_type)] = scorecard(table[table["문항 유형"] == item_type])
    cards[(criterion_name, "전체")] = scorecard(table)
cards[("늘 pass 판사", "전체")] = scorecard(teacher_table.assign(판사="pass"))
cards = pd.DataFrame(cards).T
cards.index.names = ["기준", "문항 유형"]
display(cards)

- 교사 규칙의 실패 탐지율·정상 통과율이 모호한 기준과 같거나 높고 하나라도 높다면, 기준 문장만 바꿔 오류가 줄어든 것입니다.
- 단답과 서술의 지표가 크게 다르다면 한 문항 유형에 오류가 몰린 것입니다. 5절에서 답안 유형으로 좁힙니다.

## 5. 놓친 실패·헛경보를 답안 유형별로 세세요

성적표는 얼마나 틀렸는지만 알려 주므로, 어떤 답안에서 틀렸는지 교사 규칙 판사의 틀린 칸을 답안 유형별로 셉니다.

**작성:** 함수 `error_kind(teacher, judge)`, 표 `kind_table`·`error_table`

**목적:** 교안 01 6-2의 놓친 실패(교사 fail·판사 pass)와 헛경보(교사 pass·판사 fail)를 답안 유형별 개수 표로 만듭니다.

**구현할 처리**
1. `error_kind`: 둘이 같으면 `"일치"`, 교사 fail·판사 pass면 `"놓친 실패"`, 교사 pass·판사 fail이면 `"헛경보"`를 반환합니다.
2. `teacher_table`을 복사한 `kind_table`에 행마다 `error_kind` 결과인 `구분` 열을 붙입니다.
3. `구분`이 `일치`가 아닌 행만 골라, 행이 (`문항 유형`, `답안 유형`)이고 열이 `구분`인 개수 표를 `pd.crosstab([wrong["문항 유형"], wrong["답안 유형"]], wrong["구분"])`처럼 만듭니다(`wrong`은 일치가 아닌 행). 열은 `.reindex(columns=["놓친 실패", "헛경보"], fill_value=0)`으로 맞춥니다.

**반환값:** `error_kind`는 문자열, `error_table`은 틀린 칸이 있는 답안 유형만 행으로 둔 개수 표

In [ ]:
# [작성] 판정 한 쌍을 일치·놓친 실패·헛경보로 나누는 함수와, 틀린 칸을 답안 유형별로 센 표를 만드세요.
def error_kind(teacher, judge):
    """교사·판사 판정 한 쌍을 "일치"·"놓친 실패"·"헛경보" 가운데 하나로 반환합니다."""
    # 1. 같으면 일치, 교사 fail·판사 pass면 놓친 실패, 교사 pass·판사 fail이면 헛경보
    ...


# 2. 행마다 구분 열 붙이기
kind_table = teacher_table.copy()
kind_table["구분"] = ...

# 3. 일치가 아닌 행만 골라 (문항 유형, 답안 유형) × 구분 개수 표
error_table = ...

In [ ]:
# [자가채점] 네 가지 판정 쌍으로 함수를, 하나씩 센 개수로 표를 확인합니다.
for teacher, judge, expected in [("pass", "pass", "일치"), ("fail", "fail", "일치"),
                                 ("fail", "pass", "놓친 실패"), ("pass", "fail", "헛경보")]:
    kind = error_kind(teacher, judge)
    assert kind == expected, f"교사 {teacher}·판사 {judge}: {kind!r} 대신 {expected!r}입니다. 놓친 실패는 교사 fail·판사 pass, 헛경보는 교사 pass·판사 fail입니다."

assert list(kind_table["답안 ID"]) == list(teacher_table["답안 ID"]) and "구분" in kind_table, "kind_table은 teacher_table에 구분 열을 붙인 36행 표입니다. `...`를 채웠는지 확인하세요."
expected_counts = {}
for _, row in kind_table.iterrows():
    assert row["구분"] == error_kind(row["교사"], row["판사"]), f"{row['답안 ID']}: 구분은 그 행의 error_kind(교사, 판사)입니다."
    if row["구분"] != "일치":
        key = (row["문항 유형"], row["답안 유형"], row["구분"])
        expected_counts[key] = expected_counts.get(key, 0) + 1

assert isinstance(error_table, pd.DataFrame) and list(error_table.columns) == ["놓친 실패", "헛경보"], "error_table의 열은 놓친 실패, 헛경보입니다. reindex로 열을 맞추세요."
if len(error_table):
    assert list(error_table.index.names) == ["문항 유형", "답안 유형"], "행은 (문항 유형, 답안 유형) 두 단계입니다. crosstab의 행에 두 열을 리스트로 넘기세요."
    assert (error_table.sum(axis=1) > 0).all(), "일치뿐인 답안 유형이 행에 남았습니다. 구분이 일치가 아닌 행만 골라 세세요."
for (item_type, answer_type), row in error_table.iterrows():
    for kind in ["놓친 실패", "헛경보"]:
        assert row[kind] == expected_counts.get((item_type, answer_type, kind), 0), f"{item_type} {answer_type} {kind}: 개수가 다릅니다. 일치 행을 빼고 셌는지 보세요."
assert int(error_table.to_numpy().sum()) == sum(expected_counts.values()), "일치가 아닌 칸을 모두 세세요."
print("오류 구분 확인 완료:", kind_table["구분"].value_counts().to_dict())

In [ ]:
display(error_table)

# 틀린 칸의 답안 원문과 교사 이유·판사 이유를 나란히 봅니다.
wrong_view = kind_table[kind_table["구분"] != "일치"].copy()
wrong_view["답안"] = [answer_by_id[answer_id]["answer"] for answer_id in wrong_view["답안 ID"]]
wrong_view["교사 이유"] = [answer_by_id[answer_id]["teacher_reason"] for answer_id in wrong_view["답안 ID"]]
display(wrong_view[["구분", "답안 ID", "답안", "교사 이유", "이유"]].rename(columns={"이유": "판사 이유"}))

틀린 칸마다 교사 이유와 판사 이유를 나란히 읽고 원인을 고릅니다(교안 01 6-2: 기준 문장 모호·판사 실수·교사 판정 오류). 같은 답안 유형에 몰렸다면 그 유형을 다루는 `steps` 문장부터 봅니다.

## 6. 서술형 장황함을 양방향 Pairwise로 점검하세요

서술형 판사가 긴 답에 끌리면 A만 길게 설명하고 B가 빠진 답이 짧은 정답을 이길 수 있으므로, 쌍 6개를 두 순서로 비교합니다.

| 용도 | 쌍 | a | b | 교사 승자 |
|---|---|---|---|---|
| 길이만 다름 | PE1·PE3·PE5 | 모범_정답(긴 정답) | 짧은_정답 | 무승부 |
| 긴 오답 | PE2·PE4·PE6 | 길지만_B_빠짐(A만 충족) | 짧은_정답(A·B 충족) | 짧은_정답 |

비교 프롬프트 `pair_prompt`(교안 01 4-1과 같은 모양, 기준은 3절의 `teacher_criterion`)와 `two_way`(교안 01 5-3)는 아래 셀에서 제공합니다.

In [ ]:
# 서술형 쌍 비교 프롬프트: 평가 기준·승부 규칙·문항과 답안 두 개를 줍니다.
pair_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "당신은 같은 문항에 대한 학생 답안 두 개를 비교하는 판사입니다. "
        "주어진 평가 기준과 승부 규칙만 적용합니다. 답안의 순서와 길이는 승부 이유가 아닙니다. "
        "이유를 먼저 쓰고, 기준에서 차이가 없으면 tie를 고르세요."
    )),
    ("human", "평가 기준: {criterion}\n승부 규칙: {pair_rule}\n\n"
              "문항: {question}\n\n[답안 A]\n{answer_a}\n\n[답안 B]\n{answer_b}"),
])
pair_chain = pair_prompt | judge_llm.with_structured_output(PairVerdict)


def two_way(first, second):
    """양방향 판정: 두 순서의 승자가 같으면 그 승자를, 다르면 "무승부"를 반환합니다."""
    return first if first == second else "무승부"


pairs = quiz["pairs"]
print("승부 규칙:", quiz["pair_rule"])
display(pd.DataFrame(pairs).rename(columns={"pair_id": "쌍 ID", "item_id": "문항 ID", "purpose": "용도", "teacher_winner": "교사 승자"}))

**작성:** 함수 `to_answer_id(pair, winner, swapped)`, 입력 목록 `first_inputs`·`swapped_inputs`, 표 `pair_table`

**목적:** 교안 01 5-2·5-3처럼 자리를 바꿔 두 번 묻고, 승자를 답안 ID로 되돌려 양방향 판정을 냅니다.

**구현할 처리**
1. `to_answer_id`: 첫 순서(`swapped=False`)는 A→a, B→b, 바꾼 순서(`swapped=True`)는 A→b, B→a로 되돌립니다. `tie`는 `"무승부"`입니다.
2. 쌍마다 첫 순서 입력(a를 A 자리)을 만듭니다. `criterion`=`teacher_criterion(문항)`, `pair_rule`=`quiz["pair_rule"]`, `question`=문항의 `question`, `answer_a`·`answer_b`=a·b 답안 원문.
3. 바꾼 순서 입력은 첫 순서 입력에서 `answer_a`·`answer_b`만 바꿉니다.
4. (다음 [작성] 셀) 두 입력 목록을 각각 `pair_chain.batch`로 판정합니다(판사 호출 6+6번).
5. 쌍마다 행을 만들어 `pair_table`로 모읍니다. 열은 `쌍 ID`, `용도`, `a`, `b`, `교사 승자`, `첫 순서`, `바꾼 순서`, `양방향`이고, `양방향`은 `two_way(첫 순서, 바꾼 순서)`입니다.

**반환값:** `to_answer_id`는 답안 ID 또는 `"무승부"`, `pair_table`은 6행 표

In [ ]:
# [작성] 승자를 답안 ID로 되돌리는 함수와 두 순서의 입력 목록을 만드세요(판사 호출 0번).
def to_answer_id(pair, winner, swapped):
    """판사의 승자(A·B·tie)를 답안 ID로 되돌립니다. swapped=True면 b가 A 자리에 있던 바꾼 순서입니다."""
    # 1. tie는 무승부, 첫 순서는 A→a·B→b, 바꾼 순서는 A→b·B→a
    ...


# 2. 첫 순서 입력 (a가 A 자리)
first_inputs = []
for pair in pairs:
    item = item_by_id[pair["item_id"]]
    first_inputs.append({
        "criterion": ...,
        "pair_rule": ...,
        "question": ...,
        "answer_a": ...,
        "answer_b": ...,
    })

# 3. 바꾼 순서 입력 (answer_a·answer_b만 바꿈)
swapped_inputs = ...

In [ ]:
# [자가채점] 가짜 승자로 되돌리기 규칙을, 입력 목록으로 두 순서를 확인합니다.
fake_pair = {"a": "X-긴_답", "b": "X-짧은_답"}
for winner, swapped, expected in [("A", False, "X-긴_답"), ("B", False, "X-짧은_답"), ("tie", False, "무승부"),
                                  ("A", True, "X-짧은_답"), ("B", True, "X-긴_답"), ("tie", True, "무승부")]:
    got = to_answer_id(fake_pair, winner, swapped)
    assert got == expected, f"승자 {winner}, swapped={swapped}: {got!r} 대신 {expected!r}입니다. 바꾼 순서에서는 b가 A 자리에 있었습니다."

assert isinstance(first_inputs, list) and isinstance(swapped_inputs, list) and len(first_inputs) == len(swapped_inputs) == len(pairs), "쌍마다 입력 하나씩, 두 목록 모두 6개입니다. `...`를 채웠는지 확인하세요."
for pair, first, swapped in zip(pairs, first_inputs, swapped_inputs):
    item = item_by_id[pair["item_id"]]
    assert set(first) == {"criterion", "pair_rule", "question", "answer_a", "answer_b"}, "입력 키는 pair_prompt의 칸 이름 다섯 개입니다."
    assert first["criterion"] == teacher_criterion(item), f"{pair['pair_id']}: criterion은 3절의 teacher_criterion(문항)입니다."
    assert first["pair_rule"] == quiz["pair_rule"] and first["question"] == item["question"], f"{pair['pair_id']}: pair_rule은 quiz['pair_rule'], question은 그 쌍 문항의 question입니다."
    assert first["answer_a"] == answer_by_id[pair["a"]]["answer"] and first["answer_b"] == answer_by_id[pair["b"]]["answer"], f"{pair['pair_id']}: 첫 순서는 a 답안 원문을 A 자리, b 답안 원문을 B 자리에 둡니다(답안 ID가 아니라 원문)."
    assert swapped["answer_a"] == first["answer_b"] and swapped["answer_b"] == first["answer_a"], f"{pair['pair_id']}: 바꾼 순서는 answer_a와 answer_b를 맞바꿉니다."
    assert all(swapped[key] == first[key] for key in ["criterion", "pair_rule", "question"]), f"{pair['pair_id']}: 바꾼 순서에서 기준·규칙·문항은 첫 순서와 같습니다."
print("되돌리기 규칙과 입력 확인 완료: 쌍", len(first_inputs), "개")

In [ ]:
# [작성] 쌍 6개를 두 순서로 판정하고(판사 호출 12번), 승자를 답안 ID로 되돌린 표를 만드세요.

# 4. 두 순서로 판정 (동시에 32개까지)
first_verdicts = ...
swapped_verdicts = ...

# 5. 쌍마다 한 행
pair_rows = []
for pair, first, second in zip(pairs, first_verdicts, swapped_verdicts):
    first_id = ...   # 첫 순서 승자(first.winner: A·B·tie)를 답안 ID로
    second_id = ...  # 바꾼 순서 승자(second.winner: A·B·tie)를 답안 ID로
    pair_rows.append({
        "쌍 ID": pair["pair_id"], "용도": pair["purpose"], "a": pair["a"], "b": pair["b"],
        "교사 승자": pair["teacher_winner"], "첫 순서": first_id, "바꾼 순서": second_id,
        "양방향": ...,
    })
pair_table = pd.DataFrame(pair_rows)
display(pair_table)

In [ ]:
# [자가채점] 표 모양과 되돌린 값을 확인합니다. 승자 자체는 실행마다 달라지므로 보지 않습니다.
pair_columns = ["쌍 ID", "용도", "a", "b", "교사 승자", "첫 순서", "바꾼 순서", "양방향"]
assert isinstance(pair_table, pd.DataFrame) and list(pair_table.columns) == pair_columns, f"pair_table의 열은 {pair_columns} 순서입니다. `...`를 채웠는지 확인하세요."
assert list(pair_table["쌍 ID"]) == [pair["pair_id"] for pair in pairs], "행은 pairs 순서로 6개입니다."
for pair, first, second, (_, row) in zip(pairs, first_verdicts, swapped_verdicts, pair_table.iterrows()):
    for column in ["첫 순서", "바꾼 순서", "양방향"]:
        assert row[column] in (pair["a"], pair["b"], "무승부"), f"{pair['pair_id']} {column}: 답안 ID(a·b) 또는 '무승부'로 되돌리세요. A·B·tie가 그대로 남았는지 보세요."
    assert row["첫 순서"] == to_answer_id(pair, first.winner, False), f"{pair['pair_id']}: 첫 순서는 first.winner(A·B·tie)를 to_answer_id(pair, first.winner, swapped=False)로 되돌립니다."
    assert row["바꾼 순서"] == to_answer_id(pair, second.winner, True), f"{pair['pair_id']}: 바꾼 순서는 second.winner(A·B·tie)를 to_answer_id(pair, second.winner, swapped=True)로 되돌립니다."
    assert row["양방향"] == two_way(row["첫 순서"], row["바꾼 순서"]), f"{pair['pair_id']}: 양방향은 two_way(첫 순서, 바꾼 순서)입니다."
print("양방향 표 확인 완료: 쌍", len(pair_table), "개")

In [ ]:
# 장황함·위치 점검: 긴 답이 이긴 횟수와, 긴 오답 쌍에서 양방향 판정이 교사 승자와 다른 쌍을 셉니다.
long_wrong = pair_table[pair_table["용도"] == "긴 오답"]      # a = 길지만_B_빠짐, 교사 승자 = 짧은_정답
length_only = pair_table[pair_table["용도"] == "길이만 다름"]  # a = 모범_정답(긴 정답), 교사 승자 = 무승부
long_pair_misses = int((long_wrong["양방향"] != long_wrong["교사 승자"]).sum())  # 무승부도 셉니다. 7절 도입 판정에 씁니다

print("긴 오답: 판정 6번 중 길지만_B_빠짐이 이긴 횟수", int(long_wrong[["첫 순서", "바꾼 순서"]].eq(long_wrong["a"], axis=0).sum().sum()),
      "| 양방향 판정이 교사 승자와 다른 쌍(long_pair_misses)", long_pair_misses, "/", len(long_wrong))
print("길이만 다름: 판정 6번 중 모범_정답이 이긴 횟수", int(length_only[["첫 순서", "바꾼 순서"]].eq(length_only["a"], axis=0).sum().sum()),
      "| 양방향 무승부", int((length_only["양방향"] == "무승부").sum()), "/", len(length_only))
print("위치: 두 순서의 승자가 다른 쌍", int((pair_table["첫 순서"] != pair_table["바꾼 순서"]).sum()), "/", len(pair_table))

- **긴 오답 줄:** 앞 숫자(길지만_B_빠짐이 이긴 횟수)가 0보다 크면 B가 빠진 긴 답을 높게 본 것입니다(장황함 효과). 뒤 숫자 `long_pair_misses`는 두 순서 중 한 번이라도 짧은 정답을 고르지 못한 쌍 수이고(무승부 포함), 7절 판정에 씁니다.
- **길이만 다름 줄:** 앞 숫자가 0보다 크면 둘 다 A·B를 충족했는데 긴 답을 고른 것입니다(교사는 무승부).
- **위치 줄:** 0보다 크면 자리에 흔들린 것입니다(위치 효과). 긴 오답 쌍이 이 때문에 양방향 무승부가 되면 긴 답이 이기지 않았어도 `long_pair_misses`에 들어갑니다. 두 순서 모두 무승부였다면 위치가 아니라 판사가 B가 빠진 것을 보지 못한 것입니다.

## 7. 도입 판정 규칙을 함수로 옮기세요

표를 보고 그때그때 정하면 다음 기수에 다시 점검할 때 기준이 달라지므로, 운영팀의 도입 규칙을 함수로 고정합니다.

**도입 규칙** (운영팀이 정한 값이며 교안에서 정한 값이 아닙니다)

| 조건 | 기준 | 이유 |
|---|---|---|
| 실패 탐지율 | 0.8 이상 | 틀린 답을 맞다고 하면 점수가 부풀고 학생이 틀린 줄 모릅니다 |
| 정상 통과율 | 0.9 이상(문항 유형마다 pass 9개 중 헛경보 0건) | 맞은 답을 틀렸다고 하면 학생이 손해 보고 이의 신청이 늘어납니다 |
| 긴 오답 쌍(서술형만) | 양방향 판정이 교사 승자(짧은_정답)와 다른 쌍 0개(무승부 포함) | 긴 답과 짧은 정답을 비교할 때 두 순서 중 한 번이라도 짧은 정답을 고르지 못하면, 판사가 길이·순서에 흔들린다는 뜻이라 서술형 1차 채점을 맡기지 않습니다 |

**작성:** 함수 `decide_adoption(card, long_pair_misses=0)`

**목적:** 교안 01 7-5의 판사 채택 규칙(실패 탐지율·정상 통과율 하한)에 Pairwise 장황함·위치 점검을 더해 판정합니다.

**구현할 처리**
1. `card_ok`: `card["실패 탐지율"] >= MIN_FAIL_CATCH` 그리고 `card["정상 통과율"] >= MIN_PASS_KEEP`인가. `card`는 성적표 한 행입니다(예: `cards.loc[("교사 규칙", "단답")]`).
2. 장황함 조건 `no_long_pair_misses`: `long_pair_misses`가 0인가. 단답은 쌍이 없어 기본값 0으로 부릅니다.
3. 둘 다 참이면 `"판사 1차 채점"`, 아니면 `"교사 채점 유지"`를 반환합니다.

**반환값:** `"판사 1차 채점"` 또는 `"교사 채점 유지"`

In [ ]:
# [작성] 성적표 한 행과 긴 오답 쌍 결과로 도입 여부를 반환하는 함수를 완성하세요.
MIN_FAIL_CATCH = 0.8  # 실패 탐지율 하한
MIN_PASS_KEEP = 0.9   # 정상 통과율 하한 (pass 9개 중 헛경보 0건)


def decide_adoption(card, long_pair_misses=0):
    """성적표 한 행과, 긴 오답 쌍 중 양방향 판정이 교사 승자와 다른 쌍 수로 도입 여부를 반환합니다."""
    # 1. 실패 탐지율과 정상 통과율이 하한 이상인가
    card_ok = ...

    # 2. 긴 오답 쌍에서 교사 승자와 다른 쌍이 없는가
    no_long_pair_misses = ...

    # 3. 둘 다 참이면 판사 1차 채점, 아니면 교사 채점 유지
    ...

In [ ]:
# [자가채점] 가짜 성적표로 규칙의 경계를 확인합니다. 실제 성적표는 쓰지 않습니다.
def fake_card(agreement, fail_catch, pass_keep, kappa):
    """자가채점용 가짜 성적표 한 행을 만듭니다(cards의 한 행과 같은 열)."""
    return pd.Series({"일치율": agreement, "실패 탐지율": fail_catch, "정상 통과율": pass_keep, "kappa": kappa})


ADOPT, KEEP = "판사 1차 채점", "교사 채점 유지"
# (이름, 가짜 성적표, 긴 오답 쌍 중 교사 승자와 다른 쌍 수, 기대 판정, 틀렸을 때 볼 규칙)
fake_decisions = [
    ("가 두 하한과 딱 같음", fake_card(0.85, 0.8, 0.9, 0.7), 0, ADOPT, "하한과 같은 값은 충족입니다(>=)."),
    ("나 놓친 실패 2건(7/9)", fake_card(0.89, 0.78, 1.0, 0.78), 0, KEEP, "실패 탐지율 0.78은 0.8보다 낮습니다."),
    ("다 헛경보 1건(8/9)", fake_card(0.94, 1.0, 0.89, 0.89), 0, KEEP, "정상 통과율 0.89는 0.9보다 낮습니다. 헛경보가 한 건이라도 있으면 유지입니다."),
    ("라 성적표는 완벽, 긴 오답 쌍 1개가 무승부", fake_card(1.0, 1.0, 1.0, 1.0), 1, KEEP, "long_pair_misses가 0보다 크면 성적표와 상관없이 유지입니다."),
    ("마 헛경보는 없지만 놓친 실패 3건(6/9)", fake_card(0.83, 0.67, 1.0, 0.67), 0, KEEP, "실패 탐지율 0.67은 0.8보다 낮습니다. 정상 통과율이 1.0이어도 유지입니다."),
    ("바 놓친 실패 1건(8/9)", fake_card(0.94, 0.89, 1.0, 0.89), 0, ADOPT, "실패 탐지율 0.89는 하한 이상이고 정상 통과율·긴 오답 쌍 조건도 충족합니다."),
]
for name, card, misses, expected, rule in fake_decisions:
    decision = decide_adoption(card, long_pair_misses=misses)
    assert decision in (ADOPT, KEEP), f"{ADOPT!r} 또는 {KEEP!r}를 return 하세요. `...`를 채웠는지 확인하세요."
    assert decision == expected, f"[{name}] {decision!r} 대신 {expected!r}이어야 합니다. {rule}"
assert decide_adoption(fake_card(1.0, 1.0, 1.0, 1.0)) == ADOPT, "long_pair_misses를 넘기지 않으면 기본값 0으로 봅니다(단답)."
print("도입 판정 함수 확인 완료: 가짜 성적표", len(fake_decisions), "개 통과")

In [ ]:
# 교사 규칙 판사의 문항 유형별 도입 판정
adoption = {
    "단답": decide_adoption(cards.loc[("교사 규칙", "단답")]),
    "서술": decide_adoption(cards.loc[("교사 규칙", "서술")], long_pair_misses=long_pair_misses),
}
for item_type, decision in adoption.items():
    card = cards.loc[("교사 규칙", item_type)]
    extra = f" | 긴 오답 쌍 중 교사 승자와 다른 쌍 {long_pair_misses}" if item_type == "서술" else ""
    print(f"{item_type}: {decision} | 실패 탐지율 {card['실패 탐지율']} | 정상 통과율 {card['정상 통과율']}{extra}")

## 8. 도입 결정 기록을 쓰세요

**작성:** 아래 마크다운 셀을 더블클릭해 괄호 안 안내를 지우고 제목과 네 칸(상태·맥락·결정·결과)을 채웁니다. 제목은 결정이 드러나게 씁니다(형식 예: "(문항 유형)은 (판정), …"). 숫자는 자신이 실행한 표와 출력에서 옮깁니다.

**목적:** 7절 판정을 숫자로 설명하고, 판정이 바뀔 조건을 남깁니다.

**설명할 내용**
1. 상태: 받아들였으면 `채택`, 더 실행해 확인한 뒤 정할 거라면 `제안`
2. 맥락: 정할 것(문항 유형별 1차 채점 위임), 점검 방법(답안 수, 두 기준, 쌍 비교, 도입 규칙). 결정은 쓰지 않습니다.
3. 결정: 7절 판정을 "…한다."로 쓰고, `근거:`에 줄마다 숫자 하나 이상을 씁니다.
    - 두 기준의 성적표 숫자(기준 문장만 바꾼 효과)
    - 놓친 실패·헛경보가 몰린 답안 유형과 그 답안 원문 한 문장
    - 장황함·위치 점검 결과(긴 오답 쌍의 `long_pair_misses`, 길이만 다름 쌍, 두 순서의 승자가 다른 쌍)
4. 결과: `얻는 것:`, `감수하는 것:`, `다시 검토할 조건:`(바뀌는 것 → 다시 할 일) 둘 이상. 예: 기준 문장 수정 → 다시 채점, 새 답안 유형 추가 → 교사 규칙부터 보강

#### 결정 기록: (결정이 드러나는 제목)

- **상태**: (제안·채택·거절·대체됨 가운데 하나)
- **맥락**: (2–3문장: 정할 문제, 점검 방법(답안 수, 두 기준, 쌍 비교, 도입 규칙). 결정은 쓰지 않음)
- **결정**: (판정을 한 문장으로: …한다)
    - 근거: (줄마다 숫자 하나 이상: 두 기준의 성적표, 오류가 몰린 답안 유형과 원문 한 문장, 장황함·위치 점검 결과)
- **결과**
    - 얻는 것: (이 결정으로 얻는 것)
    - 감수하는 것: (대신 감수하는 것)
    - 다시 검토할 조건: (바뀌는 것 → 다시 할 일)